# Predictive Modeling of Vehicle Fuel Efficiency (MPG)

## 1. Project Overview
This notebook investigates the predictive modeling of vehicle fuel efficiency, specifically targeting the **Miles Per Gallon (MPG)** metric . Using a dataset of **398 distinct vehicle entries** from the late 20th century, we aim to compare various regression architectures—ranging from baseline linear models to advanced ensembles—to understand how physical engineering attributes influence fuel consumption.

## 2. Initial Data Inspection
The dataset consists of **8 explanatory features**, including continuous variables (displacement, weight, acceleration), discrete variables (cylinders, model year), and a nominal categorical variable (origin). 

### Data Integrity & Quality Assessment
As we saw on the EDA the **'horsepower'** feature contains **6 missing values**. These must be addressed before modeling to ensure algorithmic stability.


In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = sns.load_dataset('mpg')

print("--- PREVIEW OF FIRST 5 ROWS ---")
display(df.head())

print("\n--- DATASET INFORMATION ---")
df.info()

print("\n--- STATISTICAL SUMMARY ---")
display(df.describe())


--- PREVIEW OF FIRST 5 ROWS ---


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino



--- DATASET INFORMATION ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    object 
 8   name          398 non-null    object 
dtypes: float64(4), int64(3), object(2)
memory usage: 28.1+ KB

--- STATISTICAL SUMMARY ---


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
count,398.000000,398.000000,398.000000,392.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,104.469388,2970.424623,15.568090,76.010050
std,7.815984,1.701004,104.269838,38.491160,846.841774,2.757689,3.697627
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000
25%,17.500000,4.000000,104.250000,75.000000,2223.750000,13.825000,73.000000
50%,23.000000,4.000000,148.500000,93.500000,2803.500000,15.500000,76.000000
75%,29.000000,8.000000,262.000000,126.000000,3608.000000,17.175000,79.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000


### 1. Verify the specific count of missing values
From our initial inspection, horsepower showed 392 non-null entries vs 398 total. We then use the median value of the column to maintain the integrity of the 398 instances

In [15]:
 
missing_hp = df['horsepower'].isnull().sum()
print(f"Missing values in horsepower before imputation: {missing_hp}")

median_hp = df['horsepower'].median()
df['horsepower'] = df['horsepower'].fillna(median_hp)

print(f"Missing values after imputation: {df['horsepower'].isnull().sum()}")

Missing values in horsepower before imputation: 6
Missing values after imputation: 0


 Following our data cleaning and EDA, we implement a tiered approach to modeling.
 We will evaluate performance using Mean Squared Error (MSE) and R^2.

### Tier 1: Baseline
 * **Linear Regression**: Standard baseline for regression tasks.

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Prepare the Data
# Using the 8 explanatory features identified in the sources
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']
X = df[features]
y = df['mpg']

# Convert 'origin' using One-Hot Encoding as identified in the EDA
X = pd.get_dummies(X, columns=['origin'], drop_first=True)

# 2. Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Train the Baseline Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 4. Make Predictions
y_pred = lr_model.predict(X_test)

# 5. Benchmarking with Standardized Metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- BASELINE LINEAR REGRESSION RESULTS ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R^2): {r2:.2f}")

# 6. Documentation for GitHub README
# This establishes the baseline for the tiered modeling strategy.
# MSE represents prediction accuracy, while R^2 shows variance explained.

--- BASELINE LINEAR REGRESSION RESULTS ---
Mean Squared Error (MSE): 8.34
R-squared (R^2): 0.84


## 7. Baseline Model Evaluation: Linear Regression

We have successfully established our performance baseline using Standard Linear Regression. By training the model on the 8 explanatory features—including the nominal categorical 'origin' variable and the imputed 'horsepower' values—we achieved the following results:

### Performance Metrics:
*   **Mean Squared Error (MSE): 8.34**
*   **Coefficient of Determination ($R^2$): 0.84**

### The Error Margin (MSE 8.34)
An MSE of 8.34 indicates a Root Mean Squared Error (RMSE) of approximately 2.89 MPG. Given that our target variable ranges from 9 to 46.6 MPG, a ~3 MPG error margin leaves room for significant improvement through non-linear modeling.

### Systematic Weaknesses
- **Linear Bias**: Our EDA (Weight vs. MPG) revealed a curved trajectory that a linear model cannot capture.
- **Redundancy**: The high Pearson correlations (e.g., 0.93 between weight and displacement) suggest that a simple linear fit is mathematically unstable.

### Key Observations:
1.  **Linear Approximation**: The model assumes a strictly linear relationship between features. However, as identified in our **Bivariate Analysis (Weight vs. MPG)**, the data follows a non-linear, curved trajectory. This likely accounts for the 16% of variance the model failed to capture.
2.  **Multicollinearity Impact**: Standard Linear Regression does not account for the high pairwise correlations identified in our **Pearson Correlation Matrix** (such as the 0.93 correlation between weight and displacement). This redundancy can lead to unstable coefficients, potentially reducing the model's ability to generalize to new data.
3.  **Path Forward**: To improve these results, we will now proceed to **Polynomial Regression** to address the non-linear trajectories and **Regularized Models (Ridge/Lasso)** to mitigate the effects of multicollinearity.